In [16]:
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline

In [17]:
flower_data = pd.read_csv("Iris.csv")
flower_data["Species"].unique() # ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
flower_data.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [18]:
# pre process and feature engineering

# we dont use one hot encoding when your labels (y) have categorical values, use it when X has categorical values
# we can use .map to do numerical encoding in this case cuz we just have 3 categories but what if we have 50 categories
# we use LabelEncoder which is an automatic version of the .map function
# LabelEncoder scans a column, finds all the unique text categories and assigns a number (0, 1, 2....) to each one alphabetically

le = LabelEncoder()
flower_data['Species'] = le.fit_transform(flower_data['Species'])

X = flower_data.drop(columns = ["Species"])
y = flower_data["Species"]
y.sample(10) 

106    2
22     0
26     0
48     0
25     0
47     0
59     1
123    2
0      0
61     1
Name: Species, dtype: int64

In [19]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.5, random_state = 42) # splitting data as 50-50 

## KNN MODEL

In [23]:

# building pipeline
steps = [("scaler", StandardScaler()), ("knn", KNeighborsClassifier())]
pipeline = Pipeline(steps)

# k-fold cross validation

param_grid = {"knn__n_neighbors" : [3,5,7,9,11]}

knn_classifier_cv = GridSearchCV(
    pipeline,
    param_grid,
    cv = 5,
    
)

# train the model and make predictions

knn_classifier_cv.fit(X_train, y_train)
y_pred = knn_classifier_cv.predict(X_test)

# evaluate

print("KNN model:-\n")
print("Best K found:", knn_classifier_cv.best_params_) 

print("\naccuracy : ", accuracy_score(y_test, y_pred))

KNN model:-

Best K found: {'knn__n_neighbors': 9}

accuracy :  1.0


## Logistic regression

In [30]:
# build pipeline

steps = [("scaler", StandardScaler()), 
         ("lgr", LogisticRegression(solver='saga', max_iter=5000))] # using saga here as it supports l1 and l2 both
                                                                    # tried max iter as 1000 and 2000 but it wasnt converging 
pipeline = Pipeline(steps)

# k - fold cross validation (this time our hyper parameters are C i.e. lasso and ridge regularization)

lgr_param_grid = {
    "lgr__C" : [0.5, 1, 2, 4, 8], # trying different C values (C = 1/Lambda)
    "lgr__penalty" : ["l1", "l2"] # try all the values for both the type of penalties
}

logistic_regression_cv = GridSearchCV(
    pipeline,
    lgr_param_grid,
    cv = 5
)

# train and predict

logistic_regression_cv.fit(X_train, y_train)
y_pred = logistic_regression_cv.predict(X_test)

# evaluate

print("For Logistic Regression :-\n")
print("Best C and penalty found : ",  logistic_regression_cv.best_params_)
print("\naccuracy : ", accuracy_score(y_test, y_pred))

For Logistic Regression :-

Best C and penalty found :  {'lgr__C': 2, 'lgr__penalty': 'l1'}

accuracy :  1.0


## Naive Bayes

In [33]:
# no scaling needed in this one as we are dealing with probabilites and not distance
# no need for pipeline or gridsearch cv as well

# make model, train and predict

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

# evaluate

print("For Naive Bayes:-\n")
print("\naccuracy : ", accuracy_score(y_test, y_pred))

For Naive Bayes:-


accuracy :  1.0
